In [14]:
import os
import pandas as pd
import numpy as np
import joblib

from xgboost import XGBRegressor

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("Libraries loaded successfully")

Libraries loaded successfully


In [15]:
DATA_DIR = r"C:\Users\diksh\OneDrive\Desktop\RailWise\SIH_26027_Final_Dataset"

task_filename = None

for file in os.listdir(DATA_DIR):
    if file.lower().endswith(".csv"):
        path = os.path.join(DATA_DIR, file)
        temp = pd.read_csv(path, nrows=5)

        if "maintenance_priority_score" in temp.columns:
            task_filename = file
            break

print("Task file:", task_filename)

tasks = pd.read_csv(
    os.path.join(DATA_DIR, task_filename)
)

print("Shape:", tasks.shape)

Task file: unified_maintenance.csv
Shape: (14400, 21)


In [16]:
features = [
    "department",
    "asset_type",
    "corridor_id",
    "location_km",
    "criticality_1_5",
    "safety_risk_1_5",
    "operational_impact_1_5",
    "overdue_days",
    "estimated_duration_min",
    "required_team_size",
    "possession_required",
    "maintenance_type",
    "severity"
]

target = "maintenance_priority_score"

X = tasks[features].copy()
y = tasks[target].copy()

print("X:", X.shape)
print("y:", y.shape)

X: (14400, 13)
y: (14400,)


In [17]:
for col in X.select_dtypes(include=np.number).columns:
    X[col] = X[col].fillna(X[col].median())

for col in X.select_dtypes(exclude=np.number).columns:
    X[col] = X[col].fillna("Unknown")

print("Missing values:", X.isnull().sum().sum())

Missing values: 0


In [18]:
categorical_cols = [
    "department",
    "asset_type",
    "corridor_id",
    "possession_required",
    "maintenance_type",
    "severity"
]

X = pd.get_dummies(
    X,
    columns=categorical_cols,
    drop_first=True
)

print("Encoded feature shape:", X.shape)

Encoded feature shape: (14400, 140)


In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (11520, 140)
Testing: (2880, 140)


In [20]:
xgb_model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

print("Training XGBoost...")

xgb_model.fit(
    X_train,
    y_train
)

print("XGBoost training completed!")

Training XGBoost...
XGBoost training completed!


In [21]:
y_pred_xgb = xgb_model.predict(X_test)

print("Predictions generated:", len(y_pred_xgb))

Predictions generated: 2880


In [22]:
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2_xgb = r2_score(y_test, y_pred_xgb)

print("XGBOOST PERFORMANCE")
print("=" * 40)
print(f"MAE  : {mae_xgb:.3f}")
print(f"RMSE : {rmse_xgb:.3f}")
print(f"R²   : {r2_xgb:.4f}")

XGBOOST PERFORMANCE
MAE  : 2.118
RMSE : 2.619
R²   : 0.9654


In [23]:
print("MODEL COMPARISON")
print("=" * 50)
print(f"Random Forest | MAE: 2.518 | RMSE: 3.118 | R²: 0.9510")
print(f"XGBoost       | MAE: {mae_xgb:.3f} | RMSE: {rmse_xgb:.3f} | R²: {r2_xgb:.4f}")

MODEL COMPARISON
Random Forest | MAE: 2.518 | RMSE: 3.118 | R²: 0.9510
XGBoost       | MAE: 2.118 | RMSE: 2.619 | R²: 0.9654


In [24]:
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": xgb_model.feature_importances_
})

importance = importance.sort_values(
    "importance",
    ascending=False
)

display(importance.head(20))

,feature,importance
2,safety_risk_1_5,0.368066
4,overdue_days,0.136231
138,severity_Low,0.100900
139,severity_Medium,0.055577
3,operational_impact_1_5,0.045228
137,severity_High,0.029002
1,criticality_1_5,0.008842
136,maintenance_type_Preventive,0.007183
89,corridor_id_C00056,0.004686
44,corridor_id_C00011,0.004661


In [25]:
MODEL_DIR = r"C:\Users\diksh\OneDrive\Desktop\RailWise\models"

os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(
    xgb_model,
    os.path.join(MODEL_DIR, "xgboost_maintenance_priority_model.pkl")
)

joblib.dump(
    list(X.columns),
    os.path.join(MODEL_DIR, "model_features.pkl")
)

print("XGBoost model saved successfully!")

XGBoost model saved successfully!
